**Country:** Sri Lanka

**Season:** Maha (OND, production year = following year), Yala (AMJ,production year = same year)

**DATA**

Weather data:
1. ENSO ( nino3.4 - NOAA)
2. RAINFALL (ERA5 1979-2025 precipitation data)
3. IOD ( dmi- NOAA https://psl.noaa.gov/data/timeseries/month/DMI/ )
4. Crop production data (Department of Census & Statistics):
https://www.statistics.gov.lk/Agriculture/StaticalInformation/PaddyStatistics

**Pipeline**
* Download data
* Seasonal aggregation (means for climate indices, sum for rainfall)
* Detrending via 3rd-order polynomial for production
* OLS regressions (ENSO/DMI→rain, rain→prod, ENSO/DMI→prod direct)
* Random Forest regression & classification with time-series CV
* Bayesian logistic regression (PyMC) for differnt loss threshold (> 2.5%, 5%,7%,10%) probability
* Demo application of the risk model


In [ ]:
%pip install xarray netCDF4
%pip install pymc
import xarray as xr
import pandas as pd
import numpy as np

Defaulting to user installation because normal site-packages is not writeable
  Using cached netcdf4-1.7.4-cp311-abi3-win_amd64.whl.metadata (2.1 kB)
  Using cached cftime-1.6.5-cp313-cp313-win_amd64.whl.metadata (8.8 kB)
Using cached netcdf4-1.7.4-cp311-abi3-win_amd64.whl (21.3 MB)
Using cached cftime-1.6.5-cp313-cp313-win_amd64.whl (465 kB)

   -------------------------- ------------- 2/3 [netCDF4]
   -------------------------- ------------- 2/3 [netCDF4]
   -------------------------- ------------- 2/3 [netCDF4]
   -------------------------- ------------- 2/3 [netCDF4]
   -------------------------- ------------- 2/3 [netCDF4]
   ---------------------------------------- 3/3 [netCDF4]

Defaulting to user installation because normal site-packages is not writeable
  Using cached pymc-5.28.4-py3-none-any.whl.metadata (17 kB)
  Using cached arviz-0.23.4-py3-none-any.whl.metadata (9.1 kB)
  Using cached pytensor-2.38.2-cp313-cp313-win_amd64.whl.metadata (7.1 kB)
  Using cached h5netcdf-1.8.

Data Loading

In [2]:
# Imports
import pandas as pd
import xarray as xr

rain_path = r"E:\RIMES\ENSO_IOD_FINAL\Dataset\era5_TP_1979-2026_1deg.nc"   
nino_path = r"E:\RIMES\ENSO_IOD_FINAL\Dataset\nina34.anom.csv"
iod_path  = r"E:\RIMES\ENSO_IOD_FINAL\Dataset\dmi.had.long.csv"


# NetCDF (correct use of xarray)
ds_rain = xr.open_dataset(rain_path)

# CSV files (correct use of pandas)
nino34_df = pd.read_csv(nino_path, skipinitialspace=True)
iod_df    = pd.read_csv(iod_path, skipinitialspace=True)


print("Rain Dataset:")
print(ds_rain)

print("\nNino34 Data:")
print(nino34_df.head())

print("\nIOD Data:")
print(iod_df.head())

Rain Dataset:
<xarray.Dataset> Size: 147MB
Dimensions:     (valid_time: 564, latitude: 181, longitude: 360)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 5kB 1979-01-01T06:00:00 ... 2025-...
  * latitude    (latitude) float64 1kB 90.0 89.0 88.0 87.0 ... -88.0 -89.0 -90.0
  * longitude   (longitude) float64 3kB -180.0 -179.0 -178.0 ... 178.0 179.0
    number      int64 8B ...
    expver      (valid_time) <U4 9kB ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 147MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-21T07:11 GRIB to CDM+CF via cfgrib-0.9.1...

Nino34 Data:
         Date  \
0  1948-01-01   
1  1948-02-01   
2  1948-03-01   
3  1948-04-01   
4  1948-05-01   

   Nino Anom 3.

DATA PROCESS

In [3]:
import pandas as pd
import numpy as np
import xarray as xr

START, END = 1979, 2024

SEASONS = {
    'Maha': {'months':[10,11,12], 'offset':1},
    'Yala': {'months':[4,5,6], 'offset':0}
}

# RAINFALL PROCESSING]

rain = ds_rain['tp'] * 1000

sl_rain = rain.sel(
    latitude=slice(9.9, 5.8),
    longitude=slice(79.5, 82.0)
)

# Area-weighted spatial mean
weights = np.cos(np.deg2rad(sl_rain['latitude']))
rain_ts = sl_rain.weighted(weights).mean(dim=['latitude','longitude'])

df_rain = rain_ts.to_dataframe().reset_index()
df_rain.rename(columns={'valid_time': 'time'}, inplace=True)
df_rain['time'] = pd.to_datetime(df_rain['time'])

#  Convert Mean Daily Rate to Monthly Cumulative Total

df_rain['tp_cumulative'] = df_rain['tp'] * df_rain['time'].dt.daysinmonth

df_rain['Year'] = df_rain['time'].dt.year
df_rain['Month'] = df_rain['time'].dt.month


# ENSO PREP

nino34_df = nino34_df.iloc[:, [0, 1]].copy()
nino34_df.columns = ['Date', 'ENSO']
nino34_df['Date'] = pd.to_datetime(nino34_df['Date'])
nino34_df['Year'] = nino34_df['Date'].dt.year
nino34_df['Month'] = nino34_df['Date'].dt.month


# IOD PREP

iod_df = iod_df.iloc[:, [0, 1]].copy()
iod_df.columns = ['Date', 'IOD']
iod_df['Date'] = pd.to_datetime(iod_df['Date'])
iod_df['Year'] = iod_df['Date'].dt.year
iod_df['Month'] = iod_df['Date'].dt.month


#SEASONAL RAINFALL
def seasonal_rain(df, months):
    #Filter by season months
    temp = df[df['Month'].isin(months)].copy()

    #Group by year and sum the CUMULATIVE column
    return temp.groupby('Year')['tp_cumulative'].sum().reset_index().rename(columns={'tp_cumulative': 'tp'})


#SEASONAL ENSO

def seasonal_enso(df, months, offset=0):
    temp = df[df['Month'].isin(months)].copy()
    return temp.groupby('Year')['ENSO'].mean().reset_index()


#SEASONAL IOD

def seasonal_iod(df, months, offset=0):
    temp = df[df['Month'].isin(months)].copy()
    return temp.groupby('Year')['IOD'].mean().reset_index()


def build_season(months, offset, label):
    rain_season = seasonal_rain(df_rain, months)
    enso_season = seasonal_enso(nino34_df, months)
    iod_season  = seasonal_iod(iod_df, months)

    df = rain_season.merge(enso_season, on='Year', how='inner')
    df = df.merge(iod_season, on='Year', how='inner')

    df['Season'] = label
    return df.sort_values('Year')


# FINAL DATASETS

maha_climate_df = build_season(SEASONS['Maha']['months'], SEASONS['Maha'], 'Maha')
yala_climate_df = build_season(SEASONS['Yala']['months'], SEASONS['Yala'], 'Yala')

maha_climate_df.to_csv('maha_final_climate_rainfall.csv', index=False)
yala_climate_df.to_csv('yala_final_climate_rainfall.csv', index=False)

print(maha_climate_df.head())
print(yala_climate_df.head())

   Year          tp      ENSO       IOD Season
0  1979  768.999132  0.306667 -0.222000   Maha
1  1980  556.539965 -0.103333 -0.519000   Maha
2  1981  488.721837 -0.303333 -0.317667   Maha
3  1982  730.324719  2.036667  0.248333   Maha
4  1983  577.748774 -1.160000 -0.244000   Maha
   Year          tp      ENSO       IOD Season
0  1979  173.421438 -0.173333 -0.181000   Yala
1  1980  296.358649  0.073333 -0.181000   Yala
2  1981  294.254258 -0.520000 -0.055333   Yala
3  1982  255.091554  0.396667  0.199000   Yala
4  1983  201.835907  0.796667 -0.081333   Yala


In [5]:
import statsmodels.api as sm
import pandas as pd

maha_filtered = maha_climate_df[(maha_climate_df['Year'].astype(str).str[:4].astype(int) >= 1980) &
                           (maha_climate_df['Year'].astype(str).str[:4].astype(int) <= 2024)].copy()

yala_filtered = yala_climate_df[(yala_climate_df['Year'] >= 1980) &
                           (yala_climate_df['Year'] <= 2024)].copy()
#OLS REGRESSION FUNCTION

def run_regression_analysis(df, season_name):
    regression_results = []

    for predictor in ['ENSO', 'IOD']:
        # Prepare data (X is predictor with constant, y is rainfall)
        X = sm.add_constant(df[predictor])
        y = df['tp']

        # Fit OLS model
        model = sm.OLS(y, X).fit()

        # Store results
        regression_results.append({
            'Season': season_name,
            'Predictor': predictor,
            'Coef': model.params[predictor],
            'P_Val': model.pvalues[predictor],
            'R2': model.rsquared
        })

    return regression_results

# Run for both seasons
maha_stats = run_regression_analysis(maha_filtered, 'Maha')
yala_stats = run_regression_analysis(yala_filtered, 'Yala')

# Create the final result dataframe
results_df = pd.DataFrame(maha_stats + yala_stats)

# Format for readability
results_df['Coef'] = results_df['Coef'].round(4)
results_df['P_Val'] = results_df['P_Val'].round(4)
results_df['R2'] = results_df['R2'].round(4)

print("Regression Results:")
print(results_df)


Regression Results:
  Season Predictor      Coef   P_Val      R2
0   Maha      ENSO   66.1049  0.0047  0.1717
1   Maha       IOD  255.6181  0.0003  0.2663
2   Yala      ENSO  -19.4856  0.2863  0.0264
3   Yala       IOD   -5.8050  0.8778  0.0006


LOAD CROP DATA

In [7]:
import pandas as pd

maha_file = r"E:\RIMES\ENSO_IOD_FINAL\Dataset\PaddyExtent_Maha_Season.xlsx"
yala_file = r"E:\RIMES\ENSO_IOD_FINAL\Dataset\PaddyExtent_Yala_Season.xlsx"

maha_df = pd.read_excel(maha_file)
yala_df = pd.read_excel(yala_file)



In [8]:
import pandas as pd

#  PROCESS MAHA SEASON

maha_raw = pd.read_excel(maha_file, skiprows=4, header=None)
maha_df = maha_raw[[1, 8]].copy()
maha_df.columns = ['Year', 'Production_Bushels']


maha_df['Year'] = maha_df['Year'].astype(str).str.split('/').str[0]
maha_df['Year'] = pd.to_numeric(maha_df['Year'], errors='coerce')
maha_df['Production_Bushels'] = pd.to_numeric(maha_df['Production_Bushels'], errors='coerce') # Ensure production is numeric
maha_df = maha_df.dropna(subset=['Year', 'Production_Bushels']) # Drop NaNs from both columns
maha_df['Year'] = maha_df['Year'].astype(int) + 1  # Applied fix for all years

# Filter (1979-2024)
maha_prod = maha_df[(maha_df['Year'] >= 1979) & (maha_df['Year'] <= 2024)].reset_index(drop=True)

#  PROCESS YALA SEASON

yala_raw = pd.read_excel(yala_file, skiprows=4, header=None)
yala_df = yala_raw[[1, 8]].copy()
yala_df.columns = ['Year', 'Production_Bushels']


yala_df['Year'] = pd.to_numeric(yala_df['Year'], errors='coerce')
yala_df['Production_Bushels'] = pd.to_numeric(yala_df['Production_Bushels'], errors='coerce') # Ensure production is numeric
yala_df = yala_df.dropna(subset=['Year', 'Production_Bushels']) # Drop NaNs from both columns
yala_df['Year'] = yala_df['Year'].astype(int)

# Filter (1979-2024)
yala_prod = yala_df[(yala_df['Year'] >= 1979) & (yala_df['Year'] <= 2024)].reset_index(drop=True)


maha_prod.to_csv('maha_production_1979_2024.csv', index=False)
yala_prod.to_csv('yala_production_1979_2024.csv', index=False)

print("Maha Final:")
print(maha_prod.head())
print("\nYala Final:")
print(yala_prod.head())

Maha Final:
   Year  Production_Bushels
0  1979             66764.0
1  1980             69653.0
2  1981             72961.0
3  1982             65313.0
4  1983             85594.0

Yala Final:
   Year  Production_Bushels
0  1979             25122.0
1  1980             32584.0
2  1981             33884.0
3  1982             37999.0
4  1983             33433.0


In [9]:
import numpy as np
import pandas as pd

def perform_cubic_detrending(df, col='Production_Bushels'):
    df = df.copy()

    # Ensure correct time order
    df = df.sort_values('Year')

    # Variables
    x = df['Year'].values
    y = df[col].values

    # Center years for numerical stability
    x_centered = x - x.min()

    # Fit cubic polynomial
    coeffs = np.polyfit(x_centered, y, 3)

    # Trend estimation
    trend = np.polyval(coeffs, x_centered)

    # Store results
    df['Trend'] = trend
    df['Detrended_Production'] = y - trend

    return df


maha_detrended = perform_cubic_detrending(maha_prod)
yala_detrended = perform_cubic_detrending(yala_prod)


maha_detrended.to_csv('maha_production_detrended.csv', index=False)
yala_detrended.to_csv('yala_production_detrended.csv', index=False)

print('Maha Detrended Production:\n')
print(maha_detrended.head())
print('\nYala Detrended Production:\n')
print(yala_detrended.head())

Maha Detrended Production:

   Year  Production_Bushels         Trend  Detrended_Production
0  1979             66764.0  76649.444925          -9885.444925
1  1980             69653.0  74863.598002          -5210.598002
2  1981             72961.0  73375.089065           -414.089065
3  1982             65313.0  72172.691435          -6859.691435
4  1983             85594.0  71245.178434          14348.821566

Yala Detrended Production:

   Year  Production_Bushels         Trend  Detrended_Production
0  1979             25122.0  34496.170576          -9374.170576
1  1980             32584.0  34720.751762          -2136.751762
2  1981             33884.0  35013.709686          -1129.709686
3  1982             37999.0  35373.365597           2625.634403
4  1983             33433.0  35798.040740          -2365.040740


In [10]:
import pandas as pd
import statsmodels.api as sm

def run_seasonal_regression(climate_df, prod_df, season_label, climate_to_prod_offset=0):

    # 1. Align years
    temp_climate = climate_df.copy()
    temp_climate['Year'] = temp_climate['Year'] + climate_to_prod_offset

    # 2. Merge with detrended production
    merged = pd.merge(
        temp_climate,
        prod_df[['Year', 'Detrended_Production']],
        on='Year'
    )

    results = []
    predictors = ['ENSO', 'IOD', 'tp']

    # 3. Run regressions
    for pred in predictors:
        reg_data = merged[[pred, 'Detrended_Production']].dropna()

        if not reg_data.empty:
            X = sm.add_constant(reg_data[pred])
            y = reg_data['Detrended_Production']

            model = sm.OLS(y, X).fit()

            results.append({
                'season': season_label,
                'predictor': pred,
                'coeff': model.params[pred],
                'p val': model.pvalues[pred],
                'r2': model.rsquared
            })

    return pd.DataFrame(results)

# Execution

maha_results = run_seasonal_regression(
    maha_climate_df,
    maha_detrended,
    'Maha',
    climate_to_prod_offset=1
)

yala_results = run_seasonal_regression(
    yala_climate_df,
    yala_detrended,
    'Yala',
    climate_to_prod_offset=0
)

final_summary = pd.concat([maha_results, yala_results]).reset_index(drop=True)
print(final_summary)

  season predictor         coeff     p val        r2
0   Maha      ENSO   4278.519947  0.026401  0.109496
1   Maha       IOD  13857.738946  0.020596  0.118481
2   Maha        tp     33.465809  0.005194  0.167844
3   Yala      ENSO  -4552.951823  0.111325  0.056610
4   Yala       IOD    455.552121  0.938835  0.000135
5   Yala        tp     37.454200  0.099716  0.060392


Predict production (RF)

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. PREPARE THE DATA
# Define features t
FEATS = ['nino34_anom', 'dmi', 'rain', 'prod_lag1']


def prepare_merged_data(climate_df, prod_df, offset, season_name):
    # Align climate with production year
    c_df = climate_df.copy()
    c_df['Year'] = c_df['Year'] + offset

    # Merge on year
    df = pd.merge(c_df, prod_df[['Year', 'Detrended_Production', 'Production_Bushels']], on='Year')

    # Rename columns to match
    df = df.rename(columns={
        'ENSO': 'nino34_anom',
        'IOD': 'dmi',
        'tp': 'rain',
        'Production_Bushels': 'prod'
    })

    # Create Lagged Feature: Previous year's production shock
    df['prod_lag1'] = df['prod'].shift(1)

    return df

# Create the 'merged' dictionary used in your loop
merged = {
    ('Sri Lanka', 'Maha'): prepare_merged_data(maha_climate_df, maha_detrended, 1, 'Maha'),
    ('Sri Lanka', 'Yala'): prepare_merged_data(yala_climate_df, yala_detrended, 0, 'Yala')
}

# 2. DEFINE THE RF PIPELINE FUNCTION
def rf_reg(X, y):
    # Handle NaNs (especially from the lag feature)
    m = ~(np.isnan(y) | np.isnan(X).any(axis=1))
    X = X[m]
    y = y[m]

    if len(y) < 10:
        return None

    # Time Series Split for cross-validation
    tscv = TimeSeriesSplit(n_splits=min(5, len(y) // 3))
    preds = np.full(len(y), np.nan)
    fold = []

    for tr, te in tscv.split(X):
        # Fit on training, predict on testing
        mdl = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2).fit(X[tr], y[tr])
        p = mdl.predict(X[te])
        preds[te] = p

        fold.append(dict(
            r2=r2_score(y[te], p) if len(te) > 1 else np.nan,
            rmse=float(np.sqrt(mean_squared_error(y[te], p))),
            mae=float(mean_absolute_error(y[te], p))
        ))

    # Final model on full dataset for feature importances
    final = RandomForestRegressor(n_estimators=500, random_state=42, min_samples_leaf=2).fit(X, y)
    v = ~np.isnan(preds)

    return dict(
        n=int(len(y)),
        cv_r2_mean=float(np.nanmean([f['r2'] for f in fold])),
        cv_rmse_mean=float(np.nanmean([f['rmse'] for f in fold])),
        cv_mae_mean=float(np.nanmean([f['mae'] for f in fold])),
        oof_r2=float(r2_score(y[v], preds[v])) if v.sum() > 1 else np.nan,
        importances=final.feature_importances_
    )

# 3. RUN THE ANALYSIS
rows = []
rf_reg_results = {}

for (country, season), df in merged.items():
    # Convert to numpy arrays
    X = df[FEATS].to_numpy(float)
    y = df['prod'].to_numpy(float)

    res = rf_reg(X, y)
    rf_reg_results[(country, season)] = res

    if res is None:
        continue

    # Build result row
    r = {
        'country': country,
        'season': season,
        'model': 'RF_Regression',
        'relation': 'features->production',
        'n': res['n'],
        'cv_r2_mean': res['cv_r2_mean'],
        'cv_rmse_mean': res['cv_rmse_mean'],
        'cv_mae_mean': res['cv_mae_mean'],
        'oof_r2': res['oof_r2']
    }

    # Add importance scores for each feature
    for i, feat in enumerate(FEATS):
        r[f'imp_{feat}'] = res['importances'][i]

    rows.append(r)

# 4. OUTPUT RESULTS
results_df = pd.DataFrame(rows).round(4)
print(results_df)

# Save the merged datasets as CSVs
merged[('Sri Lanka', 'Maha')].to_csv('maha_rf_input_data.csv', index=False)
merged[('Sri Lanka', 'Yala')].to_csv('yala_rf_input_data.csv', index=False)

     country season          model              relation   n  cv_r2_mean  \
0  Sri Lanka   Maha  RF_Regression  features->production  44     -0.8543   
1  Sri Lanka   Yala  RF_Regression  features->production  45     -1.3416   

   cv_rmse_mean  cv_mae_mean  oof_r2  imp_nino34_anom  imp_dmi  imp_rain  \
0    18986.4671   16931.6858  0.3015           0.0410   0.1639    0.1771   
1    15959.6205   14184.9326  0.1988           0.0839   0.1037    0.1840   

   imp_prod_lag1  
0         0.6180  
1         0.6283  


LOSS classify (RF)

In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. DEFINE THE CLASSIFICATION FUNCTION
def rf_cls(X, y):
    # Data Cleaning: Filter out NaNs
    mask = ~(np.isnan(y) | np.isnan(X).any(axis=1))
    X_clean, y_clean = X[mask], y[mask].astype(int)

    if len(y_clean) < 10 or len(np.unique(y_clean)) < 2:
        return None

    # Time Series Cross-Validation
    tscv = TimeSeriesSplit(n_splits=min(5, len(y_clean) // 3))
    probs = np.full(len(y_clean), np.nan)
    preds = np.full(len(y_clean), np.nan)

    for tr, te in tscv.split(X_clean):
        if len(np.unique(y_clean[tr])) < 2: continue
        mdl = RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=2).fit(X_clean[tr], y_clean[tr])
        probs[te] = mdl.predict_proba(X_clean[te])[:, 1]
        preds[te] = mdl.predict(X_clean[te])

    # Final model for feature importance
    final = RandomForestClassifier(n_estimators=500, random_state=42, min_samples_leaf=2).fit(X_clean, y_clean)
    v = ~np.isnan(probs)

    try: auc = roc_auc_score(y_clean[v], probs[v]) if len(np.unique(y_clean[v])) > 1 else np.nan
    except: auc = np.nan

    return dict(n=int(len(y_clean)), accuracy=float(accuracy_score(y_clean[v], preds[v])) if v.sum() else np.nan,
                roc_auc=auc, pos_rate=float(np.mean(y_clean)), importances=final.feature_importances_)

# 2. DEFINE THE VARIABLES AND RUN THE LOOP
FEATS = ['nino34_anom', 'dmi', 'rain', 'prod_lag1']
results = []



for (country, season), df in merged.items():

    # Calculate Year-over-Year change
    df['yoy_change']  = df['prod'].pct_change()
    df['loss_any']    = (df['yoy_change'] < 0).astype(float)
    df['prod_lag1']   = df['prod'].shift(1)

    # --- Perform Random Forest Classification ---
    # We predict 'loss_any' as the primary target
    X = df[FEATS].to_numpy(float)
    y = df['loss_any'].to_numpy(float)

    res = rf_cls(X, y)
    if res:
        results.append({
            'country': country, 'season': season, 'model': 'RF_Classification',
            'target': 'loss_any', 'n': res['n'], 'accuracy': res['accuracy'],
            'roc_auc': res['roc_auc'], 'pos_rate': res['pos_rate']
        })

# 3. OUTPUT SUMMARY TABLE
rf_summary = pd.DataFrame(results).round(4)
print(rf_summary)

     country season              model    target   n  accuracy  roc_auc  \
0  Sri Lanka   Maha  RF_Classification  loss_any  44    0.6857   0.7246   
1  Sri Lanka   Yala  RF_Classification  loss_any  45    0.5714   0.6837   

   pos_rate  
0    0.3864  
1    0.4222  


BAYESIAN RISK MODEL

In [14]:
import numpy as np
import pymc as pm
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss

FEATS = ['nino34_anom', 'dmi', 'rain', 'prod_lag1']

thresholds = [2.5, 5, 7.5, 10]

bayes_results = {}

def bayes_logit(X, y):

    m = ~(np.isnan(y) | np.isnan(X).any(axis=1))
    X = X[m]
    y = y[m].astype(int)

    if len(y) < 10 or len(np.unique(y)) < 2:
        return None

    # standardization
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0
    Xs = (X - mu) / sd

    with pm.Model() as mdl:

        # stronger priors
        a = pm.Normal('intercept', 0, 0.5)
        b = pm.Normal('beta', 0, 0.5, shape=Xs.shape[1])

        p = pm.Deterministic('p', pm.math.sigmoid(a + pm.math.dot(Xs, b)))
        pm.Bernoulli('y', p=p, observed=y)

        idata = pm.sample(
            1000, tune=1000,
            chains=2, cores=1,
            progressbar=False,
            random_seed=42,
            target_accept=0.9
        )

    post = idata.posterior["p"].values.reshape(-1, len(y))
    pm_p = post.mean(axis=0)

    return {
        "n": len(y),
        "mean_risk": float(pm_p.mean()),
        "median_risk": float(np.median(pm_p)),
        "p5": float(np.percentile(pm_p, 5)),
        "p95": float(np.percentile(pm_p, 95)),
        "accuracy": float(accuracy_score(y, (pm_p >= 0.5).astype(int))),
        "roc_auc": float(roc_auc_score(y, pm_p)),
        "brier": float(brier_score_loss(y, pm_p)),
        "idata": idata,
        "mu": mu,
        "sd": sd
    }
bayes_results = {}
rows = []

for (country, season), df in merged.items():

    df['yoy_change'] = df['prod'].pct_change()
    df['prod_lag1'] = df['prod'].shift(1)

    X_base = df[FEATS].to_numpy(float)

    for thr in thresholds:

        y = (df['yoy_change'] < -thr/100).astype(float)

        print(f"Training {season} | {thr}%")

        res = bayes_logit(X_base, y)

        if res is None:
            continue

        bayes_results[(country, season, thr)] = res

        rows.append({
                "country": country,
                "season": season,
                "threshold": thr,
                "n": res["n"],
                "mean_risk": res["mean_risk"],
                "median_risk": res["median_risk"],
                "p5": res["p5"],
                "p95": res["p95"],
                "accuracy": res["accuracy"],
                "roc_auc": res["roc_auc"],
                "brier": res["brier"]
        })


results_df = pd.DataFrame(rows)

print("\n--- SUMMARY RESULTS ---")
print(results_df)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


Training Maha | 2.5%


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, beta]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 156 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Training Maha | 5%


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, beta]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 175 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


...

Training Yala | 7.5%


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, beta]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 172 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Training Yala | 10%


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, beta]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 177 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics



--- SUMMARY RESULTS ---
     country season  threshold   n  mean_risk  median_risk        p5  \
0  Sri Lanka   Maha        2.5  44   0.408662     0.429719  0.098768   
1  Sri Lanka   Maha        5.0  44   0.358220     0.349615  0.137242   
2  Sri Lanka   Maha        7.5  44   0.344508     0.338006  0.122835   
3  Sri Lanka   Maha       10.0  44   0.316110     0.303053  0.091202   
4  Sri Lanka   Yala        2.5  45   0.398069     0.360274  0.235591   
5  Sri Lanka   Yala        5.0  45   0.367813     0.337760  0.180227   
6  Sri Lanka   Yala        7.5  45   0.354540     0.312549  0.162872   
7  Sri Lanka   Yala       10.0  45   0.354540     0.312549  0.162872   

        p95  accuracy   roc_auc     brier  
0  0.756415  0.840909  0.877232  0.155084  
1  0.626797  0.795455  0.808933  0.165342  
2  0.637014  0.818182  0.825521  0.156848  
3  0.608094  0.840909  0.882353  0.131550  
4  0.633415  0.755556  0.754310  0.185551  
5  0.615005  0.822222  0.847926  0.157831  
6  0.629207  0.866

 DEMO

In [15]:
def bayes_demo_predict(bayes_results, feature_names):

    print("\nAvailable thresholds: 2.5, 5, 7.5, 10")
    threshold = float(input("Enter loss threshold (%): "))
    threshold = round(threshold, 1)

    print("Available seasons: Maha, Yala")
    season = input("Enter season: ").strip().capitalize()

    key = ("Sri Lanka", season, threshold)

    if key not in bayes_results:
        print("\nModel not found!")
        print("Available models:")
        print(list(bayes_results.keys()))
        return

    model = bayes_results[key]

    print("\nEnter feature values:")

    x_new = np.array([
        float(input(f"{f}: ")) for f in feature_names
    ])

    # standardization
    mu = model["mu"]
    sd = model["sd"]
    sd[sd == 0] = 1.0
    x_new_s = (x_new - mu) / sd

    # posterior sampling
    idata = model["idata"]

    a_samples = idata.posterior["intercept"].values.flatten()
    b_samples = idata.posterior["beta"].values.reshape(-1, len(feature_names))

    logits = a_samples + np.dot(b_samples, x_new_s)

    # stable probabilities
    probs = 1 / (1 + np.exp(-np.clip(logits, -10, 10)))

    p_mean = probs.mean()
    p5 = np.percentile(probs, 10)
    p95 = np.percentile(probs, 90)

    print("\n--- BAYESIAN PREDICTION ---")
    print(f"Season: {season}")
    print(f"Threshold: {threshold}% loss")
    print(f"Risk probability: {p_mean:.4f}")
    print(f"Uncertainty: {p95 - p5:.4f}")
bayes_demo_predict(bayes_results, FEATS)


Available thresholds: 2.5, 5, 7.5, 10


Available seasons: Maha, Yala

Enter feature values:

--- BAYESIAN PREDICTION ---
Season: Maha
Threshold: 10.0% loss
Risk probability: 0.4405
Uncertainty: 0.3995
